# 04 – Klassifikation: Vorhersage von TARGET

**CRISP-DM-Phase:** Modeling & Evaluation.
Wir trainieren vier Modelle (Baseline, Logistic Regression, Random Forest, Gradient
Boosting), vergleichen sie per Cross-Validation und Test, und – besonders wichtig –
diskutieren die für **unausgewogene** Daten geeigneten Metriken und den
Schwellenwert-Trade-off.

> **Demo-Daten-Hinweis:** Absolute Kennzahlen stammen aus synthetischen Daten und sind
> nicht inhaltlich übertragbar. Die *Methodik* und die *qualitativen Muster* (z. B.
> Imbalance-Effekt) sind aussagekräftig.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src import config, utils, preprocessing as prep, train_models as tm, evaluate_models as ev
feat = utils.load_processed("feature_matrix")
print("Feature-Matrix:", feat.shape)

Feature-Matrix: (307511, 161)


## 1. Train-Test-Split (stratifiziert)

**Warum stratifiziert:** erhält den ~10 %-Positivanteil in Train und Test. **Warum
Testset unberührt lassen:** Es dient ausschließlich der finalen, unverzerrten Schätzung
der Generalisierung. Würde man es zur Modell-/Schwellenwahl nutzen, flösse Testinformation
in die Entscheidung -> zu optimistische Ergebnisse (Overfitting auf das Testset).

In [2]:
X_train, X_test, y_train, y_test = prep.make_train_test_split(feat)
print("Train:", X_train.shape, "| Test:", X_test.shape)

18:21:14 | INFO    | Split: Train=246008 (Positivrate 0.081), Test=61503 (Positivrate 0.081)


Train: (246008, 159) | Test: (61503, 159)


## 2. Cross-Validation auf den Trainingsdaten

**Was:** 5-fache stratifizierte CV mit **PR-AUC** (Average Precision) als Leitmetrik.
**Warum CV:** robustere Schätzung als ein einzelner Split; nutzt die Trainingsdaten
effizient. **Warum PR-AUC:** fokussiert die seltene Positivklasse (Saito & Rehmsmeier,
2015). **Was ist Overfitting:** wenn ein Modell Trainingsrauschen lernt und auf neuen
Daten schlechter wird. **Erkennung:** große Lücke zwischen Trainings- und
Validierungsleistung bzw. hohe CV-Streuung.

In [3]:
cv_results = tm.cross_validate_models(X_train, y_train, scoring="average_precision")
cv_df = pd.DataFrame([(k, v[0], v[1]) for k, v in cv_results.items()],
                     columns=["model", "cv_pr_auc_mean", "cv_pr_auc_std"]
            ).sort_values("cv_pr_auc_mean", ascending=False)
utils.save_table(cv_df.round(4), "cv_results", subdir="modeling")
cv_df

18:21:15 | INFO    | Demo-Subsampling: 246008 → 40000 Zeilen (stratifiziert)


18:21:15 | INFO    | Feature-Typen: 143 numerisch, 16 kategorial.


18:21:24 | INFO    | CV dummy          average_precision = 0.0807 +/- 0.0001


18:21:39 | INFO    | CV logreg         average_precision = 0.2185 +/- 0.0115


18:21:54 | INFO    | CV random_forest  average_precision = 0.1983 +/- 0.0046


18:22:14 | INFO    | CV hist_gbdt      average_precision = 0.2255 +/- 0.0137


18:22:24 | INFO    | CV lightgbm       average_precision = 0.2293 +/- 0.0127


18:22:24 | INFO    | Tabelle gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\tables\modeling\cv_results.csv


,model,cv_pr_auc_mean,cv_pr_auc_std
4,lightgbm,0.229307,0.012683
3,hist_gbdt,0.225540,0.013652
1,logreg,0.218528,0.011541
2,random_forest,0.198312,0.004634
0,dummy,0.080725,0.000050


## 3. Modelle fitten & auf dem Testset vergleichen

**Was:** alle Modelle auf den vollen Trainingsdaten fitten, dann EINMALIG auf dem
Testset evaluieren (alle Metriken). **Interpretation der Imbalance-Falle:** Achte auf
Modelle mit hoher Accuracy, aber Recall ≈ 0 – sie sagen praktisch nie "Ausfall" voraus
und sind trotz "guter" Accuracy nutzlos für die eigentliche Aufgabe.

In [4]:
fitted = tm.fit_all_models(X_train, y_train)
comparison = ev.compare_models(fitted, X_test, y_test)
comparison.round(4)

18:22:24 | INFO    | Feature-Typen: 143 numerisch, 16 kategorial.


18:22:31 | INFO    | Gefittet: dummy


18:22:57 | INFO    | Gefittet: logreg


18:23:32 | INFO    | Gefittet: random_forest


C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] Das System kann die angegebene Datei nicht finden
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 247, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_win32()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\.venv\Lib\site-packages\joblib\externals\loky\backend\context.py", line 299, in _count_physical_cores_win32
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\dev\Lib\subproces

...

18:24:32 | INFO    | random_forest  | Acc=0.919 P=0.468 R=0.007 F1=0.015 ROC=0.740 PR=0.217


18:24:33 | INFO    | hist_gbdt      | Acc=0.920 P=0.616 R=0.026 F1=0.050 ROC=0.775 PR=0.269


C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
18:24:34 | INFO    | lightgbm       | Acc=0.714 P=0.176 R=0.694 F1=0.281 ROC=0.775 PR=0.268


18:24:34 | INFO    | Tabelle gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\tables\modeling\model_comparison.csv


,model,threshold,accuracy,precision,recall,f1,roc_auc,pr_auc
0,hist_gbdt,0.5,0.9201,0.6161,0.0262,0.0502,0.7748,0.2685
1,lightgbm,0.5,0.7139,0.1765,0.6937,0.2813,0.7751,0.2684
2,logreg,0.5,0.6982,0.1677,0.6914,0.2700,0.7629,0.2463
3,random_forest,0.5,0.9192,0.4684,0.0075,0.0147,0.7398,0.2169
4,dummy,0.5,0.9193,0.0000,0.0000,0.0000,0.5000,0.0807


**Warum Accuracy hier täuscht:** Der `dummy`-Klassifikator (immer Mehrheitsklasse)
erreicht ~90 % Accuracy bei Recall 0 – er erkennt keinen einzigen Ausfall. Genau deshalb
sind **Recall, F1 und PR-AUC** die relevanten Größen.

## 4. ROC- und PR-Kurven

**Interpretation:** ROC zeigt Rangordnungsgüte über alle Schwellen; PR fokussiert die
Positivklasse. Bei Imbalance ist die PR-Kurve aussagekräftiger – die gestrichelte Linie
markiert das Zufallsniveau (= Positivrate).

In [5]:
fig = ev.plot_roc_pr_curves(fitted, X_test, y_test); plt.show()

C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


18:24:39 | INFO    | Abbildung gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\figures\modeling\roc_pr_curves.png


C:\Users\annis\AppData\Local\Temp\ipykernel_45800\4061997854.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig = ev.plot_roc_pr_curves(fitted, X_test, y_test); plt.show()


## 5. Der Schwellenwert-Trade-off (zentral!)

**Was:** Wir variieren die Entscheidungsschwelle des besten Modells und beobachten
Precision/Recall/F1. **Warum:** Der Default 0.5 ist bei Imbalance oft ungeeignet. **Ethische
Lesart:** Eine niedrigere Schwelle erkennt mehr Ausfälle (weniger False Negatives =
weniger riskante Kredite bewilligt), erzeugt aber mehr False Positives (mehr kreditwürdige
Personen fälschlich abgelehnt). Welcher Fehler schwerer wiegt, ist eine Wert- und
Geschäftsentscheidung, keine technische Konstante.

In [6]:
best_name = comparison[comparison.model != "dummy"].iloc[0]["model"]
print("Bestes Modell (nach PR-AUC):", best_name)
thr = ev.threshold_analysis(fitted[best_name], X_test, y_test, best_name)
thr

Bestes Modell (nach PR-AUC): hist_gbdt


18:24:41 | INFO    | Threshold-Analyse (hist_gbdt):
 threshold  precision  recall    f1  predicted_positive
      0.50      0.616   0.026 0.050                 211
      0.30      0.412   0.162 0.232                1948
      0.20      0.309   0.319 0.314                5125
      0.15      0.257   0.447 0.326                8647
      0.10      0.199   0.619 0.302               15415


18:24:41 | INFO    | Tabelle gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\tables\modeling\threshold_analysis_hist_gbdt.csv


,threshold,precision,recall,f1,predicted_positive
0,0.50,0.616114,0.026183,0.050232,211
1,0.30,0.411704,0.161531,0.232027,1948
2,0.20,0.309073,0.319033,0.313974,5125
3,0.15,0.256621,0.446928,0.326036,8647
4,0.10,0.199416,0.619134,0.301668,15415


## 6. Confusion Matrix & Classification Report

In [7]:
fig, cm = ev.plot_confusion(fitted[best_name], X_test, y_test, best_name); plt.show()
print(ev.full_classification_report(fitted[best_name], X_test, y_test, best_name))

18:24:42 | INFO    | Abbildung gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\figures\modeling\confusion_hist_gbdt.png


C:\Users\annis\AppData\Local\Temp\ipykernel_45800\1856836977.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig, cm = ev.plot_confusion(fitted[best_name], X_test, y_test, best_name); plt.show()


                  precision    recall  f1-score   support

kein Ausfall (0)       0.92      1.00      0.96     56538
     Ausfall (1)       0.62      0.03      0.05      4965

        accuracy                           0.92     61503
       macro avg       0.77      0.51      0.50     61503
    weighted avg       0.90      0.92      0.88     61503



## 7. Kalibrierung (falls sinnvoll)

**Warum:** Wenn man die Wahrscheinlichkeit selbst nutzt (z. B. erwarteter Verlust),
sollte sie kalibriert sein. **Grenze:** Kalibrierung ≠ Trennschärfe.

In [8]:
fig = ev.plot_calibration(fitted[best_name], X_test, y_test, best_name); plt.show()

18:24:45 | INFO    | Abbildung gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\reports\figures\modeling\calibration_hist_gbdt.png


C:\Users\annis\AppData\Local\Temp\ipykernel_45800\4238717784.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig = ev.plot_calibration(fitted[best_name], X_test, y_test, best_name); plt.show()


## 8. Begrenztes Hyperparameter-Tuning & finales Modell

**Warum begrenzt:** großer Suchraum = teuer und überanpassungsgefährdet. **Warum auf
CV (nicht Test) getunt:** das Testset bleibt für die finale Schätzung unberührt. Das
beste Modell + Preprocessing wird als EIN Artefakt gespeichert (für die App).

In [9]:
best_est, best_params, best_score = tm.tune_best_model(X_train, y_train)
print("Beste Parameter:", best_params, "| CV PR-AUC:", round(best_score, 4))
tm.save_model(best_est, "best_model")
final_metrics = ev.evaluate_model(best_est, X_test, y_test, "best_model_tuned")
final_metrics

18:24:46 | INFO    | Demo-Subsampling: 246008 → 40000 Zeilen (stratifiziert)


18:24:46 | INFO    | Feature-Typen: 143 numerisch, 16 kategorial.


18:26:27 | INFO    | Bestes average_precision = 0.2259 bei {'classifier__learning_rate': 0.03, 'classifier__max_iter': 200, 'classifier__max_leaf_nodes': 31}


18:26:27 | INFO    | Modell gespeichert: C:\Users\annis\OneDrive - Hochschule Heilbronn\MWI\Sem1\ML\files\models\best_model.joblib


Beste Parameter: {'classifier__learning_rate': 0.03, 'classifier__max_iter': 200, 'classifier__max_leaf_nodes': 31} | CV PR-AUC: 0.2259


18:26:29 | INFO    | best_model_tuned | Acc=0.920 P=0.600 R=0.012 F1=0.024 ROC=0.760 PR=0.245


{'model': 'best_model_tuned',
 'threshold': 0.5,
 'accuracy': 0.9195974180121295,
 'precision': 0.6,
 'recall': 0.012084592145015106,
 'f1': 0.023692003948667325,
 'roc_auc': 0.7595008741547407,
 'pr_auc': 0.24500016584226572}

## 9. Fazit

- Accuracy ist bei Imbalance irreführend; **PR-AUC/Recall** sind maßgeblich.
- Der **Schwellenwert** ist eine bewusste, ethisch relevante Entscheidung.
- Das finale Modell + Preprocessing ist gespeichert.

**Nächstes Notebook (05):** unüberwachte Segmentierung (Clustering).